# Fall detector comparison

Collects the held-out results of our sequence models (`03_training`), WiFall (`04_wifall`) and FallDeFi (`05_falldefi`) into one table. It refuses runs whose split hash differs from the `03_training` run's, so every row is scored on the same test attempts.

Each approach decides on its own unit: a sequence window, a 2 s WiFall window or a FallDeFi trace. Unit metrics are therefore not directly comparable. The shared level is the attempt: an attempt is a fall when any of its units is, and it is predicted as one when any of its units is. This any-unit rule favours approaches with more units per attempt in recall and penalises them in false alarms. Sequence models are thresholded at probability 0.5, the value `03_training` reports.

**Scope.** With `SAMPLE = True` in `02_preprocess` and `03_training`, the
split covers one session (Akmal, 9 November 13:15) and 2 s windows. Results on
it are a smoke check of the pipeline, not evidence of detection performance:
the session is small, informed the baseline adaptations, and comes from one
participant.

## Runs

Set a run path to pin it; `None` takes the newest default-representation `03_training` run for `HELD_OUT_SESSION` and `KEEP_SPLIT` (`None`: the random attempt split) and the newest baseline runs fitted on its split. `EXTRA_TRAINING_RUNS` adds sequence runs on the same split, such as a representation variant, as extra rows.

In [ ]:
import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/fall_detection").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis._cache_keys import file_hash
from fall_detection.analysis.comparison import (
    baseline_group,
    choose_threshold,
    debounce,
    latest_split_run,
    latest_training_run,
    summarize,
)

ARTIFACTS = PROJECT_ROOT / "artifacts/notebooks"
# None selects the newest 03_training run for HELD_OUT_SESSION (None: random
# attempt split) and the newest baseline runs on its split; set a path to pin one.
HELD_OUT_SESSION = None
# With a holdout, datasets that kept their random split in 03_training.
KEEP_SPLIT = ("continuous_non_fall",)
TRAINING_RUN = None
WIFALL_RUN = None
FALLDEFI_RUN = None
# Extra 03_training runs on the same split, e.g. a representation variant:
# {"filtered amplitude": ARTIFACTS / "03_training/<run>"}.
EXTRA_TRAINING_RUNS = {}
# Decision rule for sequence runs that saved validation predictions: the
# threshold is chosen on validation attempts for this attempt false-positive
# rate, then applied to the test attempts with runs of k alarmed windows.
DECISION_TARGET_FALSE_POSITIVE_RATE = 0.2
DECISION_CONSECUTIVE = (1, 2)
training_run = TRAINING_RUN or latest_training_run(
    ARTIFACTS / "03_training",
    "full",
    held_out_session=HELD_OUT_SESSION,
    keep_split=KEEP_SPLIT,
)
split_path = training_run / "window_split.parquet"
split = pl.read_parquet(split_path)
split_sha256 = file_hash(split_path)
wifall_run = WIFALL_RUN or latest_split_run(ARTIFACTS / "04_wifall", split_sha256)
falldefi_run = FALLDEFI_RUN or latest_split_run(ARTIFACTS / "05_falldefi", split_sha256)
for run in (wifall_run, falldefi_run):
    manifest = json.loads((run / "manifest.json").read_text())
    if manifest["split_sha256"] != split_sha256:
        raise ValueError(f"{run} used a different split than {training_run}")
# A variant's split differs only in its signal paths.
for run in EXTRA_TRAINING_RUNS.values():
    extra = pl.read_parquet(run / "window_split.parquet")
    if not extra.drop("path").equals(split.drop("path")):
        raise ValueError(
            f"{run} used different windows or attempts than {training_run}"
        )
TEST_ATTEMPTS = {
    baseline_group(group)
    for group in split.filter(pl.col("split") == "test")["group"].unique()
}
RUN = ARTIFACTS / "06_comparison" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
training_manifest = json.loads((training_run / "manifest.json").read_text())
print("Sequence models:", training_run.relative_to(PROJECT_ROOT))
print("WiFall:", wifall_run.relative_to(PROJECT_ROOT))
print("FallDeFi:", falldefi_run.relative_to(PROJECT_ROOT))
for label, run in EXTRA_TRAINING_RUNS.items():
    print(f"Sequence models ({label}):", run.relative_to(PROJECT_ROOT))
print(
    "Datasets:",
    training_manifest.get("datasets"),
    "sample:",
    training_manifest.get("sample"),
    "test attempts:",
    len(TEST_ATTEMPTS),
)

## Results

The first table is at attempt level, the second at each approach's own unit. `unscored_test_attempts` counts test attempts from which an approach produced no unit (for example, too few usable receivers).

In [ ]:
results = []
for suffix, sequence_run in (("", training_run),) + tuple(
    (f" ({label})", run) for label, run in EXTRA_TRAINING_RUNS.items()
):
    for metrics_path in sorted((sequence_run / "full").glob("*s/*/metrics.json")):
        run = json.loads(metrics_path.read_text())
        duration = run["config"]["duration"]
        for kind, model in run["models"].items():
            predictions = pl.read_parquet(
                metrics_path.parent / model["test_predictions"]
            )
            rows = predictions.with_columns(
                (pl.col("probability") >= 0.5).cast(pl.Int64).alias("prediction")
            ).to_dicts()
            results.append(
                summarize(
                    f"{kind.upper()} {duration}s{suffix}",
                    f"{duration} s window",
                    rows,
                    TEST_ATTEMPTS,
                )
                | {"family": "sequence (ours)"}
            )
for path in sorted(wifall_run.glob("*/predictions.json")):
    rows = json.loads(path.read_text())
    results.append(
        summarize(f"WiFall {path.parent.name}", "2 s window", rows, TEST_ATTEMPTS)
        | {"family": "baseline"}
    )
for path in sorted(falldefi_run.glob("*/predictions.json")):
    rows = json.loads(path.read_text())
    results.append(
        summarize("FallDeFi SVM", "trace", rows, TEST_ATTEMPTS) | {"family": "baseline"}
    )
METRICS = ("balanced_accuracy", "precision", "recall", "f1", "false_positive_rate")
table = pl.DataFrame(
    [
        {
            "approach": result["approach"],
            "family": result["family"],
            "level": level,
            "unit": result["unit"] if level == "unit" else "attempt",
            "examples": result["units"] if level == "unit" else result["attempts"],
            **{name: result[f"{level}_metrics"][name] for name in METRICS},
            "unscored_test_attempts": len(result["unscored_test_attempts"]),
        }
        for result in results
        for level in ("unit", "attempt")
    ]
)
display(table.filter(pl.col("level") == "attempt").drop("level"))
display(table.filter(pl.col("level") == "unit").drop("level"))

## Decision rule chosen on validation

The table above scores sequence windows at probability 0.5, and an attempt alarms when any window does. Here, for runs that saved validation predictions, the threshold is chosen on the validation attempts so that at most `DECISION_TARGET_FALSE_POSITIVE_RATE` of non-fall attempts alarm. A window then alarms only when it completes a run of `k` overlapping windows at or above it (`k = 1` is a plain threshold). The test attempts never set the threshold. Validation attempts come from the training sessions, so the false-positive rate reached on the test attempts shows how far the choice drifts on a new session. FallDeFi keeps its own decision.

In [ ]:
decision_results = []
for suffix, sequence_run in (("", training_run),) + tuple(
    (f" ({label})", run) for label, run in EXTRA_TRAINING_RUNS.items()
):
    for metrics_path in sorted((sequence_run / "full").glob("*s/*/metrics.json")):
        run = json.loads(metrics_path.read_text())
        duration = run["config"]["duration"]
        for kind, model in run["models"].items():
            if "validation_predictions" not in model:
                continue  # Run recorded before validation predictions were saved.
            validation = pl.read_parquet(
                metrics_path.parent / model["validation_predictions"]
            ).to_dicts()
            test = pl.read_parquet(
                metrics_path.parent / model["test_predictions"]
            ).to_dicts()
            for consecutive in DECISION_CONSECUTIVE:
                choice = choose_threshold(
                    validation,
                    target_false_positive_rate=DECISION_TARGET_FALSE_POSITIVE_RATE,
                    consecutive=consecutive,
                )
                result = summarize(
                    f"{kind.upper()} {duration}s{suffix}",
                    f"{duration} s window",
                    debounce(
                        test, threshold=choice["threshold"], consecutive=consecutive
                    ),
                    TEST_ATTEMPTS,
                )
                decision_results.append(
                    result
                    | {
                        "consecutive": consecutive,
                        "threshold": choice["threshold"],
                        "validation_false_positive_rate": choice["false_positive_rate"],
                        "validation_recall": choice["recall"],
                    }
                )
decision_table = pl.DataFrame(
    [
        {
            "approach": result["approach"],
            "k": result["consecutive"],
            "threshold": result["threshold"],
            "validation_fpr": result["validation_false_positive_rate"],
            "validation_recall": result["validation_recall"],
            "attempts": result["attempts"],
            **{name: result["attempt_metrics"][name] for name in METRICS},
        }
        for result in decision_results
    ],
    schema_overrides={"threshold": pl.Float64},
)
display(decision_table)

## Attempt-level scores

In [ ]:
attempts = table.filter(pl.col("level") == "attempt")
labels = attempts["approach"].to_list()
positions = np.arange(len(labels))
width = 0.8 / len(METRICS)
fig, axis = plt.subplots(figsize=(max(8, len(labels) * 1.4), 4))
for offset, name in enumerate(METRICS):
    axis.bar(
        positions + (offset - (len(METRICS) - 1) / 2) * width,
        attempts[name].to_numpy(),
        width,
        label=name.replace("_", " "),
    )
axis.set_xticks(positions, labels, rotation=30, ha="right")
axis.set_ylim(0, 1)
axis.set_ylabel("Held-out attempt score")
axis.set_title(f"{len(TEST_ATTEMPTS)} shared test attempts")
axis.legend(fontsize="small", ncols=len(METRICS), loc="upper center")
fig.tight_layout()
plt.show()

## Save

Writes `comparison.csv`, `decision_rule.csv` (when any run saved validation predictions), `attempt_metrics.png` and `comparison.json` to `artifacts/notebooks/06_comparison/<run>/`.

In [ ]:
RUN.mkdir(parents=True, exist_ok=False)
table.write_csv(RUN / "comparison.csv")
if len(decision_table):
    decision_table.write_csv(RUN / "decision_rule.csv")
fig.savefig(RUN / "attempt_metrics.png", dpi=150)
(RUN / "comparison.json").write_text(
    json.dumps(
        {
            "training_run": str(training_run.relative_to(PROJECT_ROOT)),
            "wifall_run": str(wifall_run.relative_to(PROJECT_ROOT)),
            "falldefi_run": str(falldefi_run.relative_to(PROJECT_ROOT)),
            "extra_training_runs": {
                label: str(run.relative_to(PROJECT_ROOT))
                for label, run in EXTRA_TRAINING_RUNS.items()
            },
            "held_out_session": training_manifest.get("held_out_session"),
            "split_sha256": split_sha256,
            "training_manifest": training_manifest,
            "test_attempts": sorted(TEST_ATTEMPTS),
            "attempt_rule": "fall if any unit is; predicted fall if any unit is",
            "results": results,
            "decision_rule": {
                "target_false_positive_rate": DECISION_TARGET_FALSE_POSITIVE_RATE,
                "consecutive": list(DECISION_CONSECUTIVE),
                "threshold_source": "validation attempts of the 03_training split",
                "results": decision_results,
            },
        },
        indent=2,
    )
    + "\n"
)
print("Comparison:", RUN.relative_to(PROJECT_ROOT))